<a href="https://colab.research.google.com/github/GD-0305/HASTS211-Assignments---Goredema-Michael-T-R2420839/blob/main/Assignment_2_Regime_Change_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Michael Tinashe Goredema - R2420839

# Time Series — Project 3
## Detecting a Regime Change in AAPL Stock Returns

This notebook applies a Hidden Markov Model to AAPL daily log returns to detect regime changes — periods where the stock switches between a calm (bull) market and a turbulent (bear) market.

**Dataset:** AAPL daily adjusted closing prices from Yahoo Finance (2018–2025)  
**Model chosen:** Regime Change Detection using a 2-state Gaussian Hidden Markov Model

In [ ]:
# install packages
!pip install yfinance hmmlearn statsmodels numpy pandas matplotlib scipy --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yfinance as yf
from hmmlearn.hmm import GaussianHMM
from statsmodels.tsa.stattools import adfuller
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('done')

---
## 1. Definition

A **regime change** in a time series refers to the changing behaviour of the time series in different time intervals. Instead of assuming a single stable process, we allow the data to switch between $K$ different states (or regimes).

We model this using a **Hidden Markov Model (HMM)**. This is a popular regime-switching model that rests on the assumption that unobserved states are determined by an underlying stochastic process known as a Markov-chain; which is a system describing possible events in which the probability of each event depends only on the state attained in the previous event.

The hidden state $S_t$ evolves over time according to a Markov chain with a transition probability matrix:

$$P(S_t = j \mid S_{t-1} = i) = p_{ij}$$

where $p_{ij}$ is the probability of switching from state $i$ to state $j$.




## 2. Description

The Hidden Markov Model assumes that the stock return on any given day comes from one of two hidden market regimes: a low-volatility state (bull market) or a high-volatility state (bear market or crisis). The model looks at the pattern of daily returns and assigns each day to the most likely regime, giving us a way to identify when the market shifted from one state to another. When markets are calm, participants are not frantically buying and selling assets so daily price returns vary less. Prices also tend to trend upward and have a low variance, suggesting a relatively low degree of spread in returns. When markets are in a state of panic, participants are frantically buying and selling assets in order to avoid losses or make short-term gains; price returns then tend to vary more, suggesting a higher variance and a higher degree of spread in returns.

In [ ]:
# download AAPL data
raw = yf.download('AAPL', start='2018-01-01', end='2025-12-31', auto_adjust=True)['Close']
raw = pd.DataFrame(raw)
raw.columns = ['Close']

# compute log returns
raw['Log_Return'] = np.log(raw['Close'] / raw['Close'].shift(1))
raw = raw.dropna()

print(f'Observations: {len(raw)}')
print(f'Date range: {raw.index[0].date()} to {raw.index[-1].date()}')
raw.head()

---
## 3. Diagram — Exploratory Plots

Before fitting the model, it helps to look at the data and check whether there are visible periods of different behaviour.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10))

# price
axes[0].plot(raw.index, raw['Close'], color='steelblue')
axes[0].set_title('AAPL Adjusted Close Price (2015–2024)')
axes[0].set_ylabel('Price (USD)')
axes[0].set_xlabel('Date')

# log returns
axes[1].plot(raw.index, raw['Log_Return'], color='darkorange', linewidth=0.7)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('AAPL Daily Log Returns')
axes[1].set_ylabel('Log Return')
axes[1].set_xlabel('Date')

# rolling 30-day volatility
rolling_vol = raw['Log_Return'].rolling(30).std() * np.sqrt(252)
axes[2].plot(raw.index, rolling_vol, color='crimson')
axes[2].set_title('AAPL 30-Day Rolling Annualised Volatility')
axes[2].set_ylabel('Annualised Volatility')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.savefig('exploratory_plots.png', dpi=120)
plt.show()
print('You can see clear spikes in volatility around early 2020 (COVID), late 2022 (rate hikes), and 2025 (geopolitical shocks)')

In [ ]:
# check if returns look normal
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(raw['Log_Return'], bins=70, density=True, color='yellow', alpha=0.6, label='AAPL returns')
x = np.linspace(raw['Log_Return'].min(), raw['Log_Return'].max(), 200)
axes[0].plot(x, stats.norm.pdf(x, raw['Log_Return'].mean(), raw['Log_Return'].std()), 'r-', lw=2, label='Normal')
axes[0].set_title('Return Distribution vs Normal')
axes[0].set_xlabel('Log Return')
axes[0].set_ylabel('Density')
axes[0].legend()

stats.probplot(raw['Log_Return'], dist='norm', plot=axes[1])
axes[1].set_title('QQ Plot of AAPL Log Returns')

plt.tight_layout()
plt.savefig('return_distribution.png', dpi=120)
plt.show()
print('The heavy tails suggest the returns are not exactly normal and come from different regimes with different volatilities')

---
## 4. Demonstration — Fitting the Model

We fit a 2-state Gaussian HMM to the AAPL log returns. The model will try to separate the data into a low-volatility regime and a high-volatility regime.

In [ ]:
# prepare data — HMM needs a 2D array
returns_array = raw['Log_Return'].values.reshape(-1, 1)

# fit the model
np.random.seed(42)
hmm_model = GaussianHMM(n_components=2, covariance_type='full', n_iter=500, random_state=42)
hmm_model.fit(returns_array)

print(f'Model converged: {hmm_model.monitor_.converged}')
print(f'Log-likelihood:  {hmm_model.score(returns_array):.4f}')

In [ ]:
# extract model parameters
means = hmm_model.means_.flatten()
vols  = np.sqrt(hmm_model.covars_.flatten())

# sort so state 0 = bull (higher mean) and state 1 = bear (lower mean)
order = np.argsort(means)[::-1]
means = means[order]
vols  = vols[order]
trans = hmm_model.transmat_[np.ix_(order, order)]

labels = ['Bull (low volatility)', 'Bear (high volatility)']

print('Estimated Model Parameters')
print('='*45)
for i in range(2):
    print(f'\nRegime {i+1}: {labels[i]}')
    print(f'  Daily mean return:     {means[i]*100:.4f}%')
    print(f'  Annualised return:     {means[i]*252*100:.2f}%')
    print(f'  Daily volatility:      {vols[i]*100:.4f}%')
    print(f'  Annualised volatility: {vols[i]*np.sqrt(252)*100:.2f}%')

print('\nTransition Probability Matrix')
trans_df = pd.DataFrame(trans,
    index=[f'From {l}' for l in labels],
    columns=[f'To {l}' for l in labels])
print(trans_df.round(4))

print('\nExpected number of days in each regime before switching:')
for i in range(2):
    duration = 1 / (1 - trans[i, i])
    print(f'  {labels[i]}: {duration:.1f} days (~{duration/21:.1f} months)')

In [ ]:
# decode the most likely sequence of regimes using Viterbi
states = hmm_model.predict(returns_array)

# remap to match our sorted order
remap = {order[0]: 0, order[1]: 1}
states = np.array([remap[s] for s in states])

raw['Regime'] = states

# posterior probabilities
posteriors = hmm_model.predict_proba(returns_array)
raw['P_Bull'] = posteriors[:, order[0]]
raw['P_Bear'] = posteriors[:, order[1]]

print('Days in each regime:')
counts = raw['Regime'].value_counts().sort_index()
for i, lbl in enumerate(labels):
    pct = counts[i] / len(raw) * 100
    print(f'  {lbl}: {counts[i]} days ({pct:.1f}%)')

In [ ]:
# plot the regimes on top of the price and returns
colors_map = {0: 'g', 1: 'r'}  # green = bull, red = bear

fig, axes = plt.subplots(3, 1, figsize=(14, 11))

# price coloured by regime
for i in range(len(raw) - 1):
    c = colors_map[raw['Regime'].iloc[i]]
    axes[0].plot(raw.index[i:i+2], raw['Close'].iloc[i:i+2], color=c, linewidth=1)
axes[0].set_title('AAPL Price — Coloured by Detected Regime')
axes[0].set_ylabel('Price (USD)')
axes[0].set_xlabel('Date')
bull_patch = mpatches.Patch(color='g', label='Bull regime')
bear_patch = mpatches.Patch(color='r', label='Bear regime')
axes[0].legend(handles=[bull_patch, bear_patch])

# returns coloured by regime
for i in range(len(raw) - 1):
    c = colors_map[raw['Regime'].iloc[i]]
    axes[1].plot(raw.index[i:i+2], raw['Log_Return'].iloc[i:i+2], color=c, linewidth=0.7, alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.6, linestyle='--')
axes[1].set_title('AAPL Log Returns — Coloured by Detected Regime')
axes[1].set_ylabel('Log Return')
axes[1].set_xlabel('Date')
axes[1].legend(handles=[bull_patch, bear_patch])

# posterior probabilities
axes[2].fill_between(raw.index, raw['P_Bull'], alpha=0.5, color='g', label='P(Bull)')
axes[2].fill_between(raw.index, raw['P_Bear'], alpha=0.5, color='r', label='P(Bear)')
axes[2].set_title('Probability of Each Regime Over Time')
axes[2].set_ylabel('Probability')
axes[2].set_xlabel('Date')
axes[2].legend()

plt.tight_layout()
plt.savefig('regime_detection.png', dpi=120)
plt.show()

### Interpreting the Parameters

From the model output above:

- **Bull regime:** The model identifies a state with a positive daily mean return and relatively low volatility. Most of 2017, 2019, and 2021 fall into this regime. The high transition probability of staying in the bull regime (the diagonal entry) means this is a persistent state — once the market is calm, it tends to stay calm. 78.2% of the time the market was bullish in nature.

- **Bear regime:** This state has a near-zero or negative daily mean and much higher volatility. The COVID crash period (Feb–April 2020) and the 2022 sell-off as well as the geopolitical shocks of 2025 are clearly captured here. The bear regime is less persistent — the market tends to recover back to the bull regime fairly quickly. The remaining 21.8% (according to the model) of the time, the market was bearish in nature.

- **Transition probabilities:** The probability of staying in the bull regime from one day to the next is high (typically above 0.97), meaning regime switches are rare events. This matches what we observe in financial markets — long calm periods with occasional sharp downturns.

---
## 5. Diagnosis — Checking the Model

In [ ]:
# ADF test — check that log returns are stationary
# (regime switching should be in the variance, not the mean level)
adf_stat, p_val, lags, nobs, crit, _ = adfuller(raw['Log_Return'], autolag='AIC')

print('ADF Test on AAPL Log Returns')
print(f'ADF statistic: {adf_stat:.4f}')
print(f'p-value:       {p_val:.6f}')
print(f'Critical values: {crit}')
print('')
if p_val < 0.05:
    print('Result: reject the unit root — log returns are stationary')
    print('This confirms the regime switching is in the variance, not the level of the series')
else:
    print('Result: cannot reject unit root')

In [ ]:
# compute standardised residuals within each regime
raw['Std_Resid'] = np.nan
for k in range(2):
    mask = raw['Regime'] == k
    raw.loc[mask, 'Std_Resid'] = (raw.loc[mask, 'Log_Return'] - means[k]) / vols[k]

# diagnostic plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# QQ plot of standardised residuals
stats.probplot(raw['Std_Resid'].dropna(), dist='norm', plot=axes[0])
axes[0].set_title('QQ Plot of Standardised Residuals')

# histogram of residuals vs standard normal
axes[1].hist(raw['Std_Resid'].dropna(), bins=60, density=True, color='steelblue',
             alpha=0.6, label='Residuals')
x = np.linspace(-4, 4, 200)
axes[1].plot(x, stats.norm.pdf(x), 'r-', lw=2, label='N(0,1)')
axes[1].set_title('Standardised Residuals vs N(0,1)')
axes[1].set_xlabel('Standardised Residual')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig('diagnostic_plots.png', dpi=120)
plt.show()

In [ ]:
# check skewness and kurtosis of residuals within each regime
print('Residual statistics within each regime:')
for k, lbl in enumerate(labels):
    resids = raw[raw['Regime'] == k]['Std_Resid'].dropna()
    print(f'  {lbl}:')
    print(f'    n = {len(resids)},  skewness = {resids.skew():.4f},  excess kurtosis = {resids.kurtosis():.4f}')
    jb, pval = stats.jarque_bera(resids)
    print(f'    Jarque-Bera: stat={jb:.2f}, p={pval:.4f}')

---
## 6. Damage — Problems the Model Reveals

From the diagnostics above, we can see a few issues:

1. **Heavy tails persist within regimes.** Even after separating the data into two regimes, the standardised residuals still show some excess kurtosis (heavy tailed in both directions). This means a normal distribution within each regime is still not a perfect fit — the model helps but does not completely resolve the heavy-tail problem.

2. **Some days are hard to classify.** On days where the posterior probability is close to 50/50, the model is genuinely uncertain about the regime. This is especially common around regime transitions.

3. **Only 2 regimes may be too simple.** The model treats all calm periods as the same, but there may be unexplained differences mid-state. A 3-state model might fit better.


---
## 7. Directions — Can We Improve the Model?

Based on the diagnostics, here are some ways we could improve the model:

1. **Use a 3-state model** if the BIC comparison favours it. A third regime could represent a transition or recovery state between bull and bear.

2. **Use Student-t emissions instead of Gaussian.** Since the residuals still show heavy tails, replacing the normal distribution with a Student-t within each regime would handle extreme returns better.

3. **Shorten the estimation window.** Fitting the model on just 2019–2025 (excluding the very different pre-2019 period) might give cleaner regimes, since market structure has changed.



---
## 8. Deployment — How Would This Model Be Used?

In practice, a regime detection model like this could be used in several ways:

1. **Adjusting position sizing:** When the model assigns a high probability to the bear regime, a portfolio manager could reduce equity exposure and move into safer assets like bonds or cash.

2. **Options pricing:** The regime-specific volatility estimates ($\sigma_1$ and $\sigma_2$) give a more realistic view of volatility than a single average. For example, if the bear regime volatility is 40% annualised, put options should be priced using something closer to that during a downturn rather than the long-run average of 25%.

3. **Risk management:** A risk manager could use the posterior bear probability as a warning indicator. If $P(\text{bear}) > 60\%$, tighten stop-losses or reduce leverage across the book.

4. **Daily monitoring:** The model could be re-estimated weekly using rolling data, and the current regime probability reported to traders each morning as part of a market risk briefing.

In [ ]:
# simple regime-aware strategy: only hold AAPL when bull probability > 0.6
raw['Signal'] = (raw['P_Bull'] > 0.6).astype(int)
raw['Strategy'] = raw['Signal'].shift(1) * raw['Log_Return']
raw['BuyHold']  = raw['Log_Return']

# cumulative returns
raw['Cumul_Strategy'] = raw['Strategy'].fillna(0).cumsum().apply(np.exp)
raw['Cumul_BuyHold']  = raw['BuyHold'].cumsum().apply(np.exp)

# simple annualised Sharpe
def sharpe(r):
    return (r.mean() / r.std()) * np.sqrt(252)

print('Strategy performance (illustrative backtest only):')
print(f'  Regime-aware total return: {raw["Cumul_Strategy"].iloc[-1]-1:.2%}')
print(f'  Buy-and-hold total return: {raw["Cumul_BuyHold"].iloc[-1]-1:.2%}')
print(f'  Regime-aware Sharpe: {sharpe(raw["Strategy"].dropna()):.3f}')
print(f'  Buy-and-hold Sharpe: {sharpe(raw["BuyHold"].dropna()):.3f}')
print('')
print('Note: this is a backtest using the full-sample Viterbi states, so it has look-ahead bias.')
print('In a real deployment, only past data would be used to estimate the current regime.')

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(raw.index, raw['Cumul_BuyHold'],  label='Buy and Hold', color='steelblue')
plt.plot(raw.index, raw['Cumul_Strategy'], label='Regime-Aware Strategy', color='darkorange', linestyle='--')
plt.title('Cumulative Return: Regime-Aware Strategy vs Buy and Hold (AAPL 2015–2024)')
plt.ylabel('Growth of $1')
plt.xlabel('Date')
plt.legend()
plt.tight_layout()
plt.savefig('deployment_strategy.png', dpi=120)
plt.show()

#Non-Technical Report
##Section 1 - What I found
After analysing Apple's daily stock returns from 2018 to 2025, the stock's behaviour clearly falls into two distinct market phases: a calm phase (2018, most of 2019 and 2021, 2023-2024) and a turbulent phase (early 2020, most of 2022 and 2025).
- Calm Phase: characterised by steady, positive growth with small daily swings. This phase dominated most of the decade.
- Turbulent Phase: characterised by erratic price movement with large daily swings. This phase was less frequent, but severely damaging when it occurred.

##Section 2 - Recommended Course of Action
- During turbulent phase: reduce Apple exposure and invest in safer assets (bonds, cash). Also apply tighter loss limits.
- During calm phases: maintain and grow AAPL positions. Since the calm phase is persistent, it will remain so long enough to reward continued investment.
- As an ongoing discipline: continue to monitor market conditions and adjust exposure depending on the market conditions.

##Section 3 - What drove the phases
- Calm phases: These were mainly influenced by strong earning, surging consumer demand, Apple services growth, and AI optimism.
- Turbulent phases: These were influenced primarily by the COVID-19 pandemic (systemic shocks) and rapid interest rate hikes as well as the US-China trade war (geopolitical shocks) in 2025.

Thus, macroeconomic factors can be found to be the dominant triggers for phase changes in Apple's stock; and not just company-specific news.

---
##References
- Hamilton, J.D. (1989). A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle. Econometrica, 57(2), 357–384.

- Rabiner, L.R. (1989). A Tutorial on Hidden Markov Models and Selected Applications in Speech Recognition. Proceedings of the IEEE, 77(2), 257–286.

- Hull, J.C. (2018). Options, Futures, and Other Derivatives (10th ed.). Pearson.

- hmmlearn documentation: https://hmmlearn.readthedocs.io

- Yahoo Finance data accessed via yfinance Python library.